In [1]:
import pandas as pd

In [52]:
df1 = pd.read_csv("../data/fuelwind_data")
df2 = pd.read_csv("../data/windfor_data")

In [85]:
df1 = df1.loc[:, ~df1.columns.str.startswith("Unnamed")]
df2 = df2.loc[:, ~df2.columns.str.startswith("Unnamed")]
df1 = df1.sort_values(by=["startTime", "publishTime"], ascending=True)
df2 = df2.sort_values(by=["startTime", "publishTime"], ascending=True)

In [93]:
df1.head()

,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation
1487,FUELHH,2026-01-01T00:00:00Z,2025-12-31T23:30:00Z,2025-12-31,48,WIND,13733
1486,FUELHH,2026-01-01T00:30:00Z,2026-01-01T00:00:00Z,2026-01-01,1,WIND,13756
1485,FUELHH,2026-01-01T01:00:00Z,2026-01-01T00:30:00Z,2026-01-01,2,WIND,13873
1484,FUELHH,2026-01-01T01:30:00Z,2026-01-01T01:00:00Z,2026-01-01,3,WIND,14036
1483,FUELHH,2026-01-01T02:00:00Z,2026-01-01T01:30:00Z,2026-01-01,4,WIND,14077


In [107]:
df2['startTime'].unique()

<StringArray>
['2025-12-24T21:00:00Z', '2025-12-24T22:00:00Z', '2025-12-24T23:00:00Z',
 '2025-12-25T00:00:00Z', '2025-12-25T01:00:00Z', '2025-12-25T02:00:00Z',
 '2025-12-25T03:00:00Z', '2025-12-25T04:00:00Z', '2025-12-25T05:00:00Z',
 '2025-12-25T06:00:00Z',
 ...
 '2026-02-02T12:00:00Z', '2026-02-02T13:00:00Z', '2026-02-02T14:00:00Z',
 '2026-02-02T15:00:00Z', '2026-02-02T16:00:00Z', '2026-02-02T17:00:00Z',
 '2026-02-02T18:00:00Z', '2026-02-02T19:00:00Z', '2026-02-02T20:00:00Z',
 '2026-02-02T21:00:00Z']
Length: 961, dtype: str

In [90]:
df1c = df1.copy()
df2c = df2.copy()
df2c.head()

,dataset,publishTime,startTime,generation
22119,WINDFOR,2025-12-25T02:30:00Z,2025-12-24T21:00:00Z,10625
22046,WINDFOR,2025-12-25T04:30:00Z,2025-12-24T21:00:00Z,10625
21973,WINDFOR,2025-12-25T07:30:00Z,2025-12-24T21:00:00Z,10625
21900,WINDFOR,2025-12-25T09:30:00Z,2025-12-24T21:00:00Z,10625
21827,WINDFOR,2025-12-25T11:30:00Z,2025-12-24T21:00:00Z,10625


In [91]:
df2c["startTime"] = pd.to_datetime(df2c["startTime"], utc=True)
df2c["publishTime"] = pd.to_datetime(df2c["publishTime"], utc=True)

# Calculate decision cutoff (24 hours before delivery)
df2c["decision_cutoff"] = (
    df2c["startTime"] - pd.Timedelta(days=1)
)

# Keep only forecasts published before the decision cutoff
df2c = df2c[
    df2c["publishTime"] <= df2c["decision_cutoff"]
]

# Keep the latest eligible forecast for each delivery interval
df2c = (
    df2c.sort_values("publishTime")
        .groupby("startTime", as_index=False)
        .tail(1)
        .sort_values("startTime")
        .reset_index(drop=True)
)

In [94]:
df2c.head()

,dataset,publishTime,startTime,generation,decision_cutoff
0,WINDFOR,2025-12-25 02:30:00+00:00,2025-12-26 03:00:00+00:00,9741,2025-12-25 03:00:00+00:00
1,WINDFOR,2025-12-25 02:30:00+00:00,2025-12-26 04:00:00+00:00,9496,2025-12-25 04:00:00+00:00
2,WINDFOR,2025-12-25 04:30:00+00:00,2025-12-26 05:00:00+00:00,9334,2025-12-25 05:00:00+00:00
3,WINDFOR,2025-12-25 04:30:00+00:00,2025-12-26 06:00:00+00:00,9402,2025-12-25 06:00:00+00:00
4,WINDFOR,2025-12-25 04:30:00+00:00,2025-12-26 07:00:00+00:00,9411,2025-12-25 07:00:00+00:00


In [98]:
df1c.head()
df1c["startTime"] = pd.to_datetime(df1c["startTime"], utc=True)
df1c["publishTime"] = pd.to_datetime(df1c["publishTime"], utc=True)

In [102]:
merged_df = pd.merge(
    df1c,
    df2c,
    on="startTime",
    how="left",
    suffixes=("_actual", "_forecast")
)
merged_df.head(15)

,dataset_actual,publishTime_actual,startTime,settlementDate,settlementPeriod,fuelType,generation_actual,dataset_forecast,publishTime_forecast,generation_forecast,decision_cutoff
0,FUELHH,2026-01-01 00:00:00+00:00,2025-12-31 23:30:00+00:00,2025-12-31,48,WIND,13733,NaN,NaT,NaN,NaT
1,FUELHH,2026-01-01 00:30:00+00:00,2026-01-01 00:00:00+00:00,2026-01-01,1,WIND,13756,WINDFOR,2025-12-30 22:30:00+00:00,19327.0,2025-12-31 00:00:00+00:00
2,FUELHH,2026-01-01 01:00:00+00:00,2026-01-01 00:30:00+00:00,2026-01-01,2,WIND,13873,NaN,NaT,NaN,NaT
3,FUELHH,2026-01-01 01:30:00+00:00,2026-01-01 01:00:00+00:00,2026-01-01,3,WIND,14036,WINDFOR,2025-12-30 22:30:00+00:00,19425.0,2025-12-31 01:00:00+00:00
4,FUELHH,2026-01-01 02:00:00+00:00,2026-01-01 01:30:00+00:00,2026-01-01,4,WIND,14077,NaN,NaT,NaN,NaT
5,FUELHH,2026-01-01 02:30:00+00:00,2026-01-01 02:00:00+00:00,2026-01-01,5,WIND,14185,WINDFOR,2025-12-30 22:30:00+00:00,19382.0,2025-12-31 02:00:00+00:00
6,FUELHH,2026-01-01 03:00:00+00:00,2026-01-01 02:30:00+00:00,2026-01-01,6,WIND,14575,NaN,NaT,NaN,NaT
7,FUELHH,2026-01-01 03:30:00+00:00,2026-01-01 03:00:00+00:00,2026-01-01,7,WIND,14942,WINDFOR,2025-12-31 02:30:00+00:00,19401.0,2025-12-31 03:00:00+00:00
8,FUELHH,2026-01-01 04:00:00+00:00,2026-01-01 03:30:00+00:00,2026-01-01,8,WIND,15264,NaN,NaT,NaN,NaT
9,FUELHH,2026-01-01 04:30:00+00:00,2026-01-01 04:00:00+00:00,2026-01-01,9,WIND,15303,WINDFOR,2025-12-31 02:30:00+00:00,19393.0,2025-12-31 04:00:00+00:00


In [110]:
merged_df.to_csv(
    "../server/data/wind_generation.csv",
    index=False
)